# Step 5 — Gemma-7B-IT inference

One of four parallel notebooks for this step (one LLM each). Each one runs
in its own Colab tab on a GPU.

| notebook | model |
|----------|-------|
| 1 | Llama-2-7B-Chat |
| 2 | Llama-2-13B-Chat |
| 3 | Mistral-7B-Instruct-v0.2 |
| **this** | **Gemma-7B-IT** |


In [1]:
# Only pin what actually affects model output:
#   transformers  — generation API
#   bitsandbytes  — 8-bit quant
#   accelerate    — device mapping
# Leaving numpy/pandas unpinned on purpose — Colab ships them compiled
# together, and force-downgrading triggers ABI mismatches ("dtype size changed").
!pip install -q transformers accelerate bitsandbytes --upgrade
!pip install -q datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.0 MB/s eta 0:00:00


In [2]:
import os, json, time
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/text-difficulty-classification'
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data')
PROMPT_DIR   = os.path.join(PROJECT_ROOT, 'outputs', 'prompt_metrics')
os.makedirs(PROMPT_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - need GPU!'}")f torch.cuda.is_available():
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


In [3]:
# ---- the one cell that differs across the four notebooks ----
CURRENT_MODEL = 'gemma-7b'
model_id      = 'google/gemma-7b-it'
print(f"Model: {CURRENT_MODEL} ({model_id})")


Model: gemma-7b (google/gemma-7b-it)


In [4]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
df_all   = pd.concat([df_train, df_test], ignore_index=True)
df_all['split'] = ['train'] * len(df_train) + ['test'] * len(df_test)

with open(os.path.join(PROMPT_DIR, 'prompt_questions.json')) as f:
    ALL_PROMPTS = json.load(f)

print(f"Texts: {len(df_all)} | Prompts: {len(ALL_PROMPTS)} | Total calls: {len(df_all)*len(ALL_PROMPTS):,}")


Texts: 4548 | Prompts: 63 | Total calls: 286,524


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

print(f"Loading {CURRENT_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map='auto',
)
model.eval()
print(f"Model loaded: {CURRENT_MODEL}")


Loading gemma-7b...


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded: gemma-7b


In [6]:
# Pick a batch size by free GPU memory. These thresholds are conservative —
# Gemma-7B in 8-bit uses ~7 GB so an A100 has lots of headroom.
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_mem_gb > 35:       # A100 40/80 GB
        BATCH_SIZE = 16
    elif gpu_mem_gb > 20:     # A10G 24 GB
        BATCH_SIZE = 8
    else:                     # T4 16 GB
        BATCH_SIZE = 4
else:
    BATCH_SIZE = 1

print(f"GPU memory: {gpu_mem_gb:.1f} GB → BATCH_SIZE = {BATCH_SIZE}")




def build_prompt(text, question):
    # Exact string format matters — don't reword.
    """Build the LLM prompt for one (text, question, level) combination.
    Matches the template from Rooein et al., 2024 byte-for-byte so
    the Yes/No token distribution is reproducible.
    """
    return (
        f"Read the following text and answer the question with only 'yes' or 'no'.\n\n"
        f"Text: {text}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )


def parse_yes_no(response):
    """Turn an LLM reply into True / False / None.
    Used to aggregate Yes-rates per metric across the dataset.
    """
    response_lower = response.strip().lower()
    if response_lower.startswith('yes'):
        return 1
    elif response_lower.startswith('no'):
        return 0
    if 'yes' in response_lower:
        return 1
    elif 'no' in response_lower:
        return 0
    return 0


@torch.no_grad()
def _batch_generate(prompts, max_new_tokens=10):
    # Raw batch generation. May OOM — caller handles that.
    """Low-level: tokenize a batch of prompts, run model.generate with
    greedy decoding, and return decoded completions (prompt stripped).
    Kept private (underscore) because callers should use
    get_llm_responses_batch instead.
    """
    tokenizer.padding_side = 'left'
    inputs = tokenizer(
        prompts, return_tensors='pt', truncation=True,
        max_length=2048, padding=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=None, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    input_length = inputs['input_ids'].shape[1]
    responses = []
    for i in range(len(prompts)):
        new_tokens = outputs[i][input_length:]
        responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True))

    del inputs, outputs
    torch.cuda.empty_cache()
    return responses


@torch.no_grad()
def get_llm_responses_batch(prompts, max_new_tokens=10):
    # Try full batch, fall back to one-at-a-time if it OOMs.
    """High-level batched inference.
    Chunks `prompts` into BATCH_SIZE-sized groups, calls
    _batch_generate on each, and concatenates results in order.
    """
    try:
        return _batch_generate(prompts, max_new_tokens)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        responses = []
        for prompt in prompts:
            responses.append(_batch_generate([prompt], max_new_tokens)[0])
        return responses


# smoke test
test_resp = get_llm_responses_batch([
    build_prompt("The sun is a star.", "Is this text suitable for elementary school?")
])
print(f"Test: '{test_resp[0]}' -> {parse_yes_no(test_resp[0])}")
print(f"Batch inference ready! BATCH_SIZE={BATCH_SIZE}")


GPU memory: 42.4 GB → BATCH_SIZE = 16


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Test: ' Yes/No

The text is simple and easy' -> 1
Batch inference ready! BATCH_SIZE=16


In [7]:
# Resume from a prior run if there's a progress file
progress_path = os.path.join(PROMPT_DIR, f'{CURRENT_MODEL}_progress.csv')
if os.path.exists(progress_path):
    existing  = pd.read_csv(progress_path)
    start_idx = len(existing)
    results   = existing.to_dict('records')
    print(f"RESUMING from text {start_idx}/{len(df_all)}")
else:
    start_idx = 0
    results   = []
    print(f"Starting fresh: 0/{len(df_all)}")
print(f"Remaining: {len(df_all) - start_idx} texts")


Starting fresh: 0/4548
Remaining: 4548 texts


In [8]:
SAVE_EVERY = 50

t_start         = time.time()
texts_done      = 0
total_remaining = len(df_all) - start_idx

for text_idx in tqdm(range(start_idx, len(df_all)), desc=f'{CURRENT_MODEL} batched'):
    row  = df_all.iloc[text_idx]
    text = str(row['full_text'])
    row_results = {'text_idx': text_idx, 'split': row['split']}

    all_prompts_for_text = [build_prompt(text, q) for q in ALL_PROMPTS]

    for batch_start in range(0, len(all_prompts_for_text), BATCH_SIZE):
        batch_end       = min(batch_start + BATCH_SIZE, len(all_prompts_for_text))
        batch_prompts   = all_prompts_for_text[batch_start:batch_end]
        batch_responses = get_llm_responses_batch(batch_prompts, max_new_tokens=10)
        for j, response in enumerate(batch_responses):
            prompt_idx = batch_start + j
            row_results[f'prompt_{prompt_idx}'] = parse_yes_no(response)

    results.append(row_results)
    texts_done += 1

    if (text_idx + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(progress_path, index=False)
        elapsed   = time.time() - t_start
        rate      = texts_done / elapsed
        remaining = (total_remaining - texts_done) / rate / 60
        print(f"  Checkpoint {text_idx+1}/{len(df_all)} | {rate:.2f} texts/sec | ~{remaining:.0f} min left")

df_prompt_results = pd.DataFrame(results)
df_prompt_results.to_csv(progress_path, index=False)
print(f"\nDone! {len(df_prompt_results)} texts in {(time.time()-t_start)/60:.1f} min")

gemma-7b batched:   0%|          | 0/4548 [00:00<?, ?it/s]

  Checkpoint 50/4548 | 0.13 texts/sec | ~586 min left
  Checkpoint 100/4548 | 0.13 texts/sec | ~587 min left
  Checkpoint 150/4548 | 0.13 texts/sec | ~579 min left
  Checkpoint 200/4548 | 0.13 texts/sec | ~573 min left
  Checkpoint 250/4548 | 0.13 texts/sec | ~568 min left
  Checkpoint 300/4548 | 0.12 texts/sec | ~567 min left
  Checkpoint 350/4548 | 0.12 texts/sec | ~566 min left
  Checkpoint 400/4548 | 0.12 texts/sec | ~559 min left
  Checkpoint 450/4548 | 0.12 texts/sec | ~550 min left
  Checkpoint 500/4548 | 0.12 texts/sec | ~541 min left
  Checkpoint 550/4548 | 0.12 texts/sec | ~534 min left
  Checkpoint 600/4548 | 0.12 texts/sec | ~528 min left
  Checkpoint 650/4548 | 0.12 texts/sec | ~524 min left
  Checkpoint 700/4548 | 0.12 texts/sec | ~518 min left
  Checkpoint 750/4548 | 0.12 texts/sec | ~512 min left
  Checkpoint 800/4548 | 0.12 texts/sec | ~507 min left
  Checkpoint 850/4548 | 0.12 texts/sec | ~500 min left
  Checkpoint 900/4548 | 0.12 texts/sec | ~493 min left
  Checkpoin

In [9]:
# Split back into train/test and attach labels
prompt_cols = [c for c in df_prompt_results.columns if c.startswith('prompt_')]

train_mask = df_prompt_results['split'] == 'train'
test_mask  = df_prompt_results['split'] == 'test'

train_prompts = df_prompt_results.loc[train_mask, prompt_cols].reset_index(drop=True)
test_prompts  = df_prompt_results.loc[test_mask,  prompt_cols].reset_index(drop=True)

train_prompts['education_level'] = df_train['education_level'].values
test_prompts['education_level']  = df_test['education_level'].values

train_path = os.path.join(PROMPT_DIR, f'train_prompt_metrics_{CURRENT_MODEL}.csv')
test_path  = os.path.join(PROMPT_DIR, f'test_prompt_metrics_{CURRENT_MODEL}.csv')
train_prompts.to_csv(train_path, index=False)
test_prompts.to_csv(test_path,   index=False)

print(f"Saved: {train_prompts.shape} train, {test_prompts.shape} test")
print(f"  {train_path}")
print(f"  {test_path}")
print(f"\n{CURRENT_MODEL} COMPLETE!")


Saved: (3638, 64) train, (910, 64) test
  /content/drive/MyDrive/text-difficulty-classification/outputs/prompt_metrics/train_prompt_metrics_gemma-7b.csv
  /content/drive/MyDrive/text-difficulty-classification/outputs/prompt_metrics/test_prompt_metrics_gemma-7b.csv

gemma-7b COMPLETE!
